In [10]:
# 🔧 تنظیمات اولیه
import numpy as np
import time
import tracemalloc
# import torch
import os
import csv



In [13]:
# 1️⃣ ضرب ماتریس با حلقه‌های تودرتو (Naive - CPU)

def matmul_naive(A, B,n):
    tracemalloc.start()
    start = time.time()
    Cn = len(A)
    m = len(B[0])
    p = len(B)
    result = [[0.0 for _ in range(m)] for _ in range(n)]
    for i in range(n):
        for j in range(m):
            for k in range(p):
                result[i][j] += A[i][k] * B[k][j]
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return (end - start),(peak / 1024 / 1024),(2 * n**3 / (end - start))



In [14]:
def matmul_numpy (A_np, B_np,n):
    tracemalloc.start()
    start = time.time()
    C_numpy = np.matmul(A_np, B_np)
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return (end - start),(peak / 1024 / 1024),(2 * n**3 / (end - start))



In [15]:
# def matmul_pytorch(n):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # A_torch = torch.randn((n, n), device=device)
    # B_torch = torch.randn((n, n), device=device)
    #
    # torch.cuda.synchronize() if device.type == "cuda" else None
    # start = time.time()
    # C_torch = torch.matmul(A_torch, B_torch)
    # torch.cuda.synchronize() if device.type == "cuda" else None
    # end = time.time()
    #
    # return (end - start),(torch.cuda.memory_allocated() / 1024 ** 2),(2 * n**3 / (end - start))


In [30]:
def matmul(n):
    A_np = np.random.rand(n, n)
    B_np = np.random.rand(n, n)
    results = []
    m = n
    # نسخه Naive
    if(n > 400): 
        m = 400
        A_list = np.random.rand(m, m).tolist()
        B_list = np.random.rand(m, m).tolist()
        time_naive, mem_naive, flops_naive = matmul_naive(A_list, B_list, m)
        results.append(["Naive_CPU_MatrixMultiply", time_naive * n / m, mem_naive, f"{flops_naive:.2e}",2 * n**3])

    # نسخه NumPy
    time_numpy, mem_numpy, flops_numpy = matmul_numpy(A_np, B_np, n)
    results.append(["NumPy_CPU_MatrixMultiply", time_numpy, mem_numpy, f"{flops_numpy:.2e}",2 * n**3])

    # نسخه PyTorch
    # time_torch, mem_torch, flops_torch = matmul_pytorch(n)
    # results.append(["PyTorch_MatrixMultiply", time_torch, mem_torch, f"{flops_torch:.2e}",2 * n**3 ])

    # -------------------------------
    # ✅ ذخیره در فایل CSV

    with open("results.csv", mode="a", newline="") as file:
        writer = csv.writer(file)
        for row in results:
            writer.writerow(row)
    
    

In [17]:
def inverse_native(A):

    tracemalloc.start()
    start = time.time()
    n = len(A)
    A = np.array(A, dtype=float)
    I = np.identity(n)
    AI = np.hstack([A, I])
    for i in range(n):
        AI[i] = AI[i] / AI[i, i]
        for j in range(n):
            if i != j:
                AI[j] = AI[j] - AI[i] * AI[j, i]
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return (end - start),(peak / 1024 / 1024),(2 * n**3/ (end - start))

In [18]:
def inverse_numpy (A_np ,n):
    tracemalloc.start()
    start = time.time()
    inv_np = np.linalg.inv(A_np)
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return (end - start),(peak / 1024 / 1024),(2 * n**3 / (end - start))



In [ ]:
# def inverse_pytorch (n):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # A_torch = torch.rand(n, n, device=device)
    #
    # torch.cuda.synchronize() if device.type == "cuda" else None
    # start = time.time()
    # inv_torch = torch.linalg.inv(A_torch)
    # torch.cuda.synchronize() if device.type == "cuda" else None
    # end = time.time()
    #
    # return (end - start),(torch.cuda.memory_allocated() / 1024 ** 2),(2 * n**3 / (end - start))


In [19]:
def inverse(n):

    A_np = np.random.rand(n, n)
    results = []
    m = n
    # نسخه Naive
    if(n > 1200): 
        m = 1200
        A_list = np.random.rand(m, m).tolist()
        time_naive, mem_naive, flops_naive = inverse_native(A_list)
        results.append(["Naive_CPU_MatrixInverse", time_naive * n / m, mem_naive, f"{flops_naive:.2e}",2 * n**3])

    # نسخه NumPy
    time_numpy, mem_numpy, flops_numpy = inverse_numpy(A_np,n)
    results.append(["NumPy_CPU_MatrixInverse", time_numpy, mem_numpy, f"{flops_numpy:.2e}",2 * n**3])

    # نسخه PyTorch
    # time_torch, mem_torch, flops_torch = inverse_pytorch(n)
    # results.append(["PyTorch_MatrixInverse", time_torch, mem_torch, f"{flops_torch:.2e}",2 * n**3])

    # -------------------------------
    # ✅ ذخیره در فایل CSV

    with open("results.csv", mode="a", newline="") as file:
        writer = csv.writer(file)
        for row in results:
            writer.writerow(row)

In [20]:
def pairwise_distance_naive(X, Y):
    tracemalloc.start()
    start = time.time()
    N, D = len(X), len(X[0])
    M = len(Y)
    D_out = np.zeros((N, M))
    for i in range(N):
        for j in range(M):
            s = 0.0
            for d in range(D):
                diff = X[i][d] - Y[j][d]
                s += diff * diff
            D_out[i][j] = s ** 0.5
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return (end - start),(peak / 1024 / 1024),(2 * N * M * D / (end - start))



In [21]:
def pairwise_distance_numpy(X_np, Y_np,N,M,D):
    tracemalloc.start()
    start = time.time()
    dists_np = np.sqrt(
        np.sum(X_np**2, axis=1)[:, np.newaxis] +
        np.sum(Y_np**2, axis=1)[np.newaxis, :] -
        2 * np.dot(X_np, Y_np.T)
    )
    end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    return (end - start),(peak / 1024 / 1024),(2 * N * M * D / (end - start))


In [22]:
# def pairwise_distance_pytorch(N,M,D):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     X_torch = torch.rand(N, D, device=device)
#     Y_torch = torch.rand(M, D, device=device)
#
#     torch.cuda.synchronize() if device.type == "cuda" else None
#     start = time.time()
#     dists_torch = torch.cdist(X_torch, Y_torch, p=2)
#     torch.cuda.synchronize() if device.type == "cuda" else None
#     end = time.time()
#
#     return (end - start),(torch.cuda.memory_allocated() / 1024 ** 2),(2 * N * M * D / (end - start))


In [23]:
def distance(N,M,D):
    results = []
    N1 = N
    M1 = M
    D1 = D
    # نسخه Naive
    if(N > 400 or M > 400 or D > 400): 
        N1 = 400
        M1 = 400
        D1 = 400
        X_list = np.random.rand(N1, D1).tolist()
        Y_list = np.random.rand(M1, D1).tolist()
        time_naive, mem_naive, flops_naive = pairwise_distance_naive(X_list,Y_list)
        results.append(["Naive_CPU_Distance", time_naive * N * M * D / N1 / M1 / D1, mem_naive,  f"{flops_naive:.2e}",2 *N * M * D])

    # نسخه NumPy
    X_np = np.random.rand(N, D)
    Y_np = np.random.rand(M, D)
    time_numpy, mem_numpy, flops_numpy = pairwise_distance_numpy(X_np,Y_np,N,M,D)
    results.append(["NumPy_CPU_Distance", time_numpy, mem_numpy,  f"{flops_numpy:.2e}",2 * N * M * D])

    # نسخه PyTorch
    # time_torch, mem_torch, flops_torch = pairwise_distance_pytorch(N,M,D)
    # results.append(["PyTorch_Distance", time_torch, mem_torch,  f"{flops_torch:.2e}",2 * N * M * D])

    # -------------------------------
    # ✅ ذخیره در فایل CSV

    with open("results.csv", mode="a", newline="") as file:
        writer = csv.writer(file)
        for row in results:
            writer.writerow(row)



In [55]:
distance(200,150,100)
inverse(300)
matmul(300)